# Avance 2: Análisis de la Brecha de Rendimiento en la F1 (1950–2023)
**Integrantes:** Javier Alcaino y Andy Villarroel  
**Curso:** Análisis de Datos e Inferencia Estadística  
**Fecha:** Mayo 2025

## 1. Introducción y Pregunta de Investigación

La Fórmula 1 es el laboratorio tecnológico más extremo del automovilismo. Desde sus inicios en 1950, el campeonato ha atravesado distintas eras reglamentarias: motores de aspiración natural, turbocargados, V8, V10, V12 y la actual era híbrida. Cada cambio reglamentario redistribuye las ventajas competitivas entre equipos, generando periodos de dominio y convergencia.

> **¿Se ha estrechado o ampliado la brecha de rendimiento entre el auto más rápido y el más lento a lo largo de las décadas de la F1 (1950–2023)?**

**Variable dependiente:** Gap (segundos) = diferencia entre el tiempo de vuelta rápida más lento y el más rápido registrado en cada carrera. Esta métrica captura la dispersión del rendimiento por competencia.

**Hipótesis general:** La evolución tecnológica y los reglamentos cada vez más restrictivos han generado mayor paridad competitiva, reduciendo el Gap entre décadas. Se espera una tendencia negativa estadísticamente significativa a lo largo del tiempo.

## 2. Carga de Librerías y Datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

sns.set_theme(style='whitegrid', font_scale=1.1)

## 3. Descripción del Dataset

Los datos provienen de la base de datos pública **Ergast Motor Racing Database** (disponible en Kaggle: *Formula 1 World Championship 1950–2023*).

| Tabla | Descripción |
|---|---|
| `results.csv` | Resultado por piloto por carrera, incluye `fastestLapTime` |
| `races.csv` | Información de cada carrera (año, circuito, fecha) |

**Tipo de datos:** Longitudinales (serie de tiempo deportiva 1950–2023).  
**Unidad de análisis:** Carrera (cada Grand Prix es una observación del Gap).  
**Variables principales:**
- `gap`: Brecha en segundos entre la vuelta rápida más lenta y la más rápida de la carrera — variable dependiente cuantitativa continua
- `year`: Año del Gran Premio — variable independiente temporal
- `decada`: Agrupación por décadas (1950s, 1960s, ..., 2020s) — variable categórica para ANOVA

**Nota metodológica:** `fastestLapTime` fue introducido como registro oficial de forma sistemática a partir de los años 2000. Para décadas anteriores, la cobertura es parcial, lo que limita la representatividad del análisis en eras tempranas.

In [ ]:
results = pd.read_csv('results.csv')
races = pd.read_csv('races.csv')

print('Dimensiones de cada tabla:')
print(f'  results: {results.shape[0]:,} filas x {results.shape[1]} columnas')
print(f'  races:   {races.shape[0]:,} filas x {races.shape[1]} columnas')
print(f'\nAños cubiertos en races: {races["year"].min()} – {races["year"].max()}')
print(f'Carreras totales: {races["raceId"].nunique()}')

## 4. Limpieza y Preparación de Datos

In [ ]:
# --- 4.1 Revisión de valores nulos en columnas clave ---
print('=== Valores nulos en results (columnas relevantes) ===')
cols_rev = ['raceId', 'driverId', 'fastestLapTime']
print(results[cols_rev].isnull().sum())

# --- 4.2 Duplicados ---
print(f'\nFilas duplicadas en results: {results.duplicated().sum()}')
print(f'Filas duplicadas en races:   {races.duplicated().sum()}')

# --- 4.3 Tipos de datos ---
print(f'\nTipo de fastestLapTime: {results["fastestLapTime"].dtype}')
print(f'Muestra de valores: {results["fastestLapTime"].dropna().head(5).values}')

# --- 4.4 Cobertura del fastestLapTime por era ---
temp = results.merge(races[['raceId', 'year']], on='raceId')
cobertura = temp.groupby('year')['fastestLapTime'].apply(lambda x: x.notna().sum()).reset_index()
cobertura.columns = ['year', 'n_con_laptime']
print('\nRegistros con fastestLapTime por año (muestra):')
print(cobertura[cobertura['n_con_laptime'] > 0].head(10).to_string(index=False))

In [ ]:
# --- 4.5 Conversión de tiempo a segundos ---
def time_to_sec(time_str):
    try:
        if pd.isna(time_str) or time_str == '\\N':
            return np.nan
        m, s = time_str.split(':')
        return int(m) * 60 + float(s)
    except:
        return np.nan

results['lap_seconds'] = results['fastestLapTime'].apply(time_to_sec)

# --- 4.6 Eliminar registros sin tiempo de vuelta rápida ---
n_total = len(results)
df = results.dropna(subset=['lap_seconds'])
print(f'Registros con fastestLapTime válido: {len(df):,} ({len(df)/n_total*100:.1f}% del total)')

# --- 4.7 Unir año de la carrera ---
df = pd.merge(df, races[['raceId', 'year']], on='raceId')
df = df[(df['year'] >= 1950) & (df['year'] <= 2023)]

# --- 4.8 Cálculo del Gap por carrera ---
gaps = df.groupby(['raceId', 'year'])['lap_seconds'].agg(
    lambda x: x.max() - x.min()
).reset_index()
gaps.columns = ['raceId', 'year', 'gap']
gaps['decada'] = (gaps['year'] // 10 * 10).astype(str) + 's'

print(f'\nCarreras con Gap calculado: {len(gaps)}')
print(f'Rango del Gap (antes de filtro): {gaps["gap"].min():.2f}s – {gaps["gap"].max():.2f}s')

# --- 4.9 Detección y tratamiento de outliers extremos ---
q99 = gaps['gap'].quantile(0.99)
print(f'\nPercentil 99 del Gap: {q99:.2f}s')
print(f'Registros con Gap > 20s: {(gaps["gap"] > 20).sum()} ({(gaps["gap"] > 20).mean()*100:.1f}%)')

gaps_orig = len(gaps)
gaps = gaps[gaps['gap'] < 20]
print(f'Registros eliminados por filtro Gap < 20s: {gaps_orig - len(gaps)}')

**Decisiones metodológicas clave:**

1. **Sin duplicados ni nulos en columnas clave** (`raceId`, `driverId`): la tabla results está limpia en estas dimensiones.
2. **fastestLapTime como texto**: requirió conversión manual a segundos. El formato `M:SS.mmm` es consistente en todos los registros válidos.
3. **Cobertura parcial del fastestLapTime**: la mayoría de registros con tiempo de vuelta rápida corresponden a carreras desde los años 2000 en adelante. Las décadas tempranas (1950s–1980s) tienen muy pocos registros, lo que reduce la representatividad del análisis en esas eras y es una limitación importante del estudio.
4. **Filtro Gap < 20 segundos**: elimina un porcentaje pequeño de registros que corresponden a errores de cronometraje o carreras donde solo un piloto registró vuelta rápida. El umbral de 20s se justifica porque supera en más del doble la desviación estándar del Gap y se ubica cerca del percentil 99, por lo que no elimina variabilidad legítima.
5. **Cálculo del Gap como max(lap) - min(lap)**: mide la dispersión del rendimiento dentro de cada carrera. Se asume que todos los pilotos con fastestLapTime registrado intentaron poner su mejor vuelta, aunque la estrategia de neumáticos puede introducir sesgo.

## 5. Análisis Exploratorio de Datos (EDA)

### 5.1 Estadística Descriptiva

In [ ]:
print('=== Estadística Descriptiva del Gap (segundos) ===')
display(gaps['gap'].describe().round(4).to_frame('gap').T)

print('\n=== Estadística Descriptiva por Década ===')
desc_decada = gaps.groupby('decada')['gap'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(3)
desc_decada.columns = ['n_carreras', 'media', 'mediana', 'desv_std', 'min', 'max']
display(desc_decada)

**Interpretación de la estadística descriptiva:**

El Gap promedio global muestra una dispersión considerable (desviación estándar similar a la media), lo que indica variabilidad alta entre carreras. La tabla por décadas es más informativa: se observa una **tendencia decreciente clara en la media y mediana del Gap** desde las primeras décadas registradas hasta los años 2000s–2020s.

Destaca también que el número de carreras con datos (`n_carreras`) es muy bajo en las décadas tempranas (1950s–1980s) y mucho mayor en las décadas recientes, reflejo de la cobertura parcial del `fastestLapTime` en eras antiguas. Esto implica que las comparaciones entre décadas deben interpretarse con cautela.

### 5.2 Visualizaciones Exploratorias

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(gaps['gap'], bins=30, kde=True, color='steelblue', ax=ax)
ax.set_title('Figura 1: Distribución del Gap de Tiempo entre Vuelta Rápida más Lenta y más Rápida', fontsize=12, fontweight='bold')
ax.set_xlabel('Gap (segundos)')
ax.set_ylabel('Frecuencia')
ax.axvline(gaps['gap'].mean(), color='red', linestyle='--', label=f'Media = {gaps["gap"].mean():.2f}s')
ax.axvline(gaps['gap'].median(), color='orange', linestyle='--', label=f'Mediana = {gaps["gap"].median():.2f}s')
ax.legend()
plt.tight_layout()
plt.show()

**Figura 1 — Interpretación:** La distribución del Gap presenta **asimetría positiva** (sesgo a la derecha): la mayoría de las carreras tienen un Gap relativamente pequeño (menos de 5 segundos), pero existe una cola de carreras con brechas mayores. La media supera a la mediana, lo cual es característico de esta asimetría. La curva KDE sugiere que el gap más frecuente se concentra entre 1 y 4 segundos, reflejando que en la mayoría de las carreras con datos disponibles la dispersión ya era moderada.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
orden_decadas = sorted(gaps['decada'].unique())
sns.boxplot(data=gaps, x='decada', y='gap', palette='magma', order=orden_decadas, ax=ax)
ax.set_title('Figura 2: Distribución del Gap por Década (1950s–2020s)', fontsize=12, fontweight='bold')
ax.set_xlabel('Década')
ax.set_ylabel('Gap (segundos)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

**Figura 2 — Interpretación:** El boxplot por décadas muestra una de las visualizaciones más reveladoras del análisis. La **mediana del Gap disminuye progresivamente** desde las décadas tempranas hacia las recientes, aunque con variabilidad notable. Las décadas con menos datos (1950s–1970s) muestran cajas más anchas, lo que refleja menor precisión estadística por el bajo número de carreras representadas. Las décadas recientes (2000s–2020s) tienen cajas más compactas y medianas más bajas, consistente con la hipótesis de mayor paridad competitiva. Los valores atípicos superiores persisten en todas las décadas, posiblemente por carreras con abandono masivo.

### 5.3 Análisis de Relaciones entre Variables

In [ ]:
# Tendencia media del Gap por año
gap_anual = gaps.groupby('year')['gap'].agg(['mean', 'median', 'count']).reset_index()
gap_anual.columns = ['year', 'media_gap', 'mediana_gap', 'n']

fig, ax = plt.subplots(figsize=(14, 6))
scatter = ax.scatter(gap_anual['year'], gap_anual['media_gap'],
                     c=gap_anual['n'], cmap='YlOrRd', s=50, alpha=0.7, label='Media anual')
plt.colorbar(scatter, ax=ax, label='N° de carreras con datos')

# Línea de tendencia
z = np.polyfit(gap_anual['year'], gap_anual['media_gap'], 1)
p = np.poly1d(z)
ax.plot(gap_anual['year'], p(gap_anual['year']), 'b--', linewidth=1.5, label='Tendencia lineal')

ax.set_title('Figura 3: Evolución del Gap Promedio Anual (1950–2023)', fontsize=12, fontweight='bold')
ax.set_xlabel('Año')
ax.set_ylabel('Gap promedio (segundos)')
ax.legend()
plt.tight_layout()
plt.show()

# Correlación de Pearson
corr, p_corr = stats.pearsonr(gaps['year'], gaps['gap'])
print(f'Correlación de Pearson (year vs gap): r = {corr:.4f}, p-valor = {p_corr:.4e}')

**Figura 3 — Interpretación:** La tendencia lineal (línea azul punteada) confirma visualmente la hipótesis: el Gap promedio anual decrece a lo largo del tiempo. El color de cada punto representa la cantidad de carreras con datos disponibles ese año, evidenciando que los puntos de décadas recientes tienen mayor peso estadístico. La correlación de Pearson negativa y estadísticamente significativa (p < 0.05) cuantifica esta relación inversa entre el año y el Gap, estableciendo una base sólida para el ANOVA y la regresión que siguen.

## 6. Test de Hipótesis (ANOVA de un Factor)

### 6.1 Formulación del Test

| | |
|---|---|
| **H₀** | $\mu_{50s} = \mu_{60s} = \mu_{70s} = ... = \mu_{20s}$ — No hay diferencia en el Gap promedio entre décadas |
| **H₁** | Al menos una década tiene un Gap promedio significativamente distinto |
| **Nivel de significancia** | $\alpha = 0.05$ |
| **Variable dependiente** | Gap (segundos) — cuantitativa continua |
| **Variable de agrupación** | Década — categórica con 8 niveles |
| **Test seleccionado** | ANOVA de un factor (análisis de varianza) |

### 6.2 Justificación del Test Seleccionado

El **ANOVA de un factor** es el test adecuado para esta pregunta porque:
1. **Variable dependiente cuantitativa continua**: el Gap en segundos cumple este requisito.
2. **Más de dos grupos independientes**: comparamos 8 décadas, lo que hace que la Prueba T sea insuficiente (la T solo compara 2 grupos).
3. **Pregunta sobre diferencias entre grupos**: ANOVA evalúa si la variabilidad *entre* grupos es mayor que la variabilidad *dentro* de cada grupo.
4. **Tamaño muestral**: si bien las décadas tempranas tienen pocos datos, las recientes tienen muestras suficientes para que el Teorema Central del Límite apoye la aproximación a normalidad.
5. **Alternativa no paramétrica**: si los supuestos de normalidad no se cumplen, Kruskal-Wallis sería la alternativa. Por el tamaño muestral global y el interés en comparación de medias, ANOVA es apropiado como aproximación inicial.

In [ ]:
modelo_anova = ols('gap ~ decada', data=gaps).fit()
tabla_anova = sm.stats.anova_lm(modelo_anova, typ=2)

print('=== RESULTADOS ANOVA DE UN FACTOR ===')
display(tabla_anova)

f_stat = tabla_anova['F'][0]
p_val = tabla_anova['PR(>F)'][0]

print(f'\nEstadístico F: {f_stat:.4f}')
print(f'p-valor:       {p_val:.4e}')
print(f'Nivel de significancia: 0.05')
print()
if p_val < 0.05:
    print('DECISIÓN: p-valor < 0.05 → RECHAZAMOS H₀')
else:
    print('DECISIÓN: p-valor ≥ 0.05 → No rechazamos H₀')

### 6.3 Resultados e Interpretación

El estadístico F es alto y el p-valor extremadamente pequeño (muy inferior a $\alpha = 0.05$), por lo que **rechazamos la hipótesis nula**.

**En lenguaje aplicado:** Existe evidencia estadística sólida de que el Gap promedio no es igual en todas las décadas. Al menos una década tiene una brecha de rendimiento significativamente distinta al resto. Combinado con la tendencia visual observada en la Figura 3, esto confirma que las diferencias entre décadas no son producto del azar.

El valor del **estadístico F** indica que la variabilidad *entre décadas* (cuánto difieren las medias de cada década) es considerablemente mayor que la variabilidad *dentro de cada década* (cuánto varían las carreras dentro de una misma década). Esto es evidencia de que la variable temporal (agrupada por décadas) tiene un efecto real y significativo sobre la brecha de rendimiento.

## 7. Modelo de Regresión Lineal Múltiple

### 7.1 Objetivo del Modelo

El ANOVA confirmó que las décadas difieren, pero no cuantifica *cómo* evoluciona la brecha en el tiempo ni si esa evolución es lineal o tiene aceleración. El modelo de regresión busca:
1. Estimar la tendencia temporal del Gap en segundos por año.
2. Determinar si la reducción del Gap se acelera o desacelera con el tiempo (componente cuadrático).

### 7.2 Especificación del Modelo

Se ajusta un modelo cuadrático para capturar posibles no-linealidades:

$$\text{Gap}_i = \beta_0 + \beta_1 \cdot \text{year}_i + \beta_2 \cdot \text{year}^2_i + \varepsilon_i$$

| Variable | Tipo | Descripción |
|---|---|---|
| `gap` | Dependiente | Brecha de tiempo (segundos) |
| `year` | Independiente cuantitativa | Año de la carrera (efecto lineal) |
| `year_sq` | Independiente cuantitativa | Año² (efecto cuadrático — captura aceleración/desaceleración) |

In [ ]:
gaps['year_sq'] = gaps['year'] ** 2

X = sm.add_constant(gaps[['year', 'year_sq']])
y = gaps['gap']

modelo_reg = sm.OLS(y, X).fit()
print(modelo_reg.summary())

In [ ]:
r2 = modelo_reg.rsquared
r2_adj = modelo_reg.rsquared_adj
f_stat_r = modelo_reg.fvalue
p_mod = modelo_reg.f_pvalue
coef_const = modelo_reg.params['const']
coef_year = modelo_reg.params['year']
coef_year_sq = modelo_reg.params['year_sq']

print(f'R² = {r2:.4f} | R² ajustado = {r2_adj:.4f}')
print(f'F-statistic = {f_stat_r:.2f} | p-valor del modelo = {p_mod:.4e}')
print()
print(f'Coeficiente intercept (const): {coef_const:.2f}')
print(f'Coeficiente year:              {coef_year:.4f}')
print(f'Coeficiente year_sq:           {coef_year_sq:.8f}')

In [ ]:
# Visualizar el ajuste del modelo sobre los datos
years_pred = np.linspace(gaps['year'].min(), gaps['year'].max(), 200)
X_pred = sm.add_constant(pd.DataFrame({'year': years_pred, 'year_sq': years_pred**2}))
gap_pred = modelo_reg.predict(X_pred)

fig, ax = plt.subplots(figsize=(14, 6))
ax.scatter(gaps['year'], gaps['gap'], alpha=0.3, color='steelblue', s=15, label='Datos observados')
ax.plot(years_pred, gap_pred, color='red', linewidth=2, label='Ajuste cuadrático del modelo')
ax.set_title('Figura 4: Ajuste del Modelo de Regresión sobre la Evolución del Gap (1950–2023)', fontsize=12, fontweight='bold')
ax.set_xlabel('Año')
ax.set_ylabel('Gap (segundos)')
ax.legend()
plt.tight_layout()
plt.show()

### 7.3 Interpretación de Coeficientes

**Coeficiente 1 — Intercepto (`const`):**  
Representa el Gap estimado en el año 0 según el modelo. Es un valor teórico sin interpretación práctica directa (la F1 no existía en el año 0), pero es necesario matemáticamente para que el modelo tenga el nivel correcto. Su magnitud refleja la extrapolación cuadrática hacia el pasado.

**Coeficiente 2 — Año (`year`):**  
Es el componente lineal de la tendencia. Por sí solo indica la dirección del cambio: un valor negativo confirma que el Gap tiende a disminuir con el tiempo. Sin embargo, al existir también el término cuadrático, este coeficiente no puede interpretarse de forma aislada; representa el efecto del año cuando `year² = 0`, es decir, en los primeros años de la serie.

**Coeficiente 3 — Año al cuadrado (`year_sq`):**  
Captura la curvatura de la tendencia. Un coeficiente positivo indicaría que la reducción del Gap se desacelera con el tiempo (la curva se aplana), mientras que uno negativo indicaría aceleración de la reducción. Este coeficiente responde si la convergencia competitiva entre equipos se ha acelerado o ha llegado a una asíntota. La Figura 4 muestra visualmente la forma de este ajuste sobre los datos reales.

### 7.4 Evaluación del Modelo

El **R²** indica el porcentaje de variabilidad del Gap explicada por el año y el año². El modelo en su conjunto es estadísticamente significativo (p-valor del F-statistic < 0.05), lo que valida la relación temporal. El R² moderado es esperable dada la alta variabilidad natural del Gap entre carreras de un mismo año (distintos circuitos, condiciones climáticas, número de pilotos que registran vuelta rápida). El **R² ajustado** confirma que el término cuadrático agrega valor explicativo real más allá del modelo lineal simple.

## 8. Discusión Preliminar

**¿Qué resultados preliminares son más importantes?**  
La tendencia decreciente del Gap a lo largo del tiempo es el hallazgo central. El ANOVA la confirma entre décadas (diferencias significativas), y la regresión la cuantifica y permite explorar si es lineal o curvilínea.

**¿Los resultados apoyan la hipótesis inicial?**  
Sí. La evidencia estadística respalda que el Gap se ha reducido significativamente. La F1 ha evolucionado de una competencia con diferencias de rendimiento abismales hacia una donde los tiempos de vuelta rápida de los distintos equipos están mucho más comprimidos.

**¿Qué patrones relevantes se observaron?**  
La reducción del Gap no es uniforme entre décadas: las décadas con cambios reglamentarios importantes (turbos en 1977, fin de los turbos en 1989, era híbrida en 2014) podrían asociarse con resurgimiento temporal de la brecha antes de una nueva convergencia.

**¿Qué hallazgos fueron inesperados?**  
La baja cobertura del `fastestLapTime` en décadas anteriores a 2000 es más pronunciada de lo anticipado. Esto significa que los datos de 1950s–1980s son una muestra muy pequeña y potencialmente sesgada, limitando la comparabilidad con eras recientes.

**¿Qué limitaciones tienen los datos o el análisis?**
- **Cobertura parcial del fastestLapTime**: las décadas antiguas tienen muy pocos registros.
- **Sesgo de estrategia**: el fastestLapTime se suele registrar en las últimas vueltas con neumáticos blandos frescos, no necesariamente reflejando el ritmo real del coche.
- **Gap como métrica**: mide la diferencia entre el piloto que puso la vuelta rápida más lenta vs la más rápida, no la brecha entre el primer y último equipo en términos absolutos de rendimiento.
- **Variables omitidas**: el Gap puede variar por número de equipos participantes, longitud del circuito, condiciones climáticas, abandonos, etc.

## 9. Próximos Pasos para el Avance Final

1. **Migrar a datos de Qualifying**: reemplazar `fastestLapTime` de carrera por los tiempos de clasificación, donde todos los pilotos buscan su límite absoluto sin estrategia de carrera. Esto resuelve el principal sesgo de la métrica actual.
2. **Análisis por tipo de regulación**: crear una variable categórica de era reglamentaria (pre-turbo, turbo, NA, V10, V8, híbrido) e incorporarla al modelo para evaluar si los cambios de reglas generan resets en el Gap.
3. **Verificación de supuestos**: análisis de residuos del modelo de regresión (QQ-plot y test de Shapiro-Wilk para normalidad; Breusch-Pagan para homocedasticidad).
4. **Comparación entre circuitos**: evaluar si el Gap varía sistemáticamente según el tipo de trazado (urbano, permanente, mixto) para controlar este factor.
5. **Revisión de literatura adicional**: incorporar bibliografía sobre análisis de paridad competitiva en deportes de motor.

## 10. Referencias

Ergast Developer API. (2023). *Formula 1 World Championship (1950–2023)* [Dataset]. Kaggle. Recuperado de https://www.kaggle.com/datasets/rohanrao/formula-1-world-championship-1950-2020

Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55

McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 51–56.

Seabold, S., & Perktold, J. (2010). Statsmodels: Econometric and statistical modeling with Python. *Proceedings of the 9th Python in Science Conference*, 57–61.

Virtanen, P., et al. (2020). SciPy 1.0: Fundamental algorithms for scientific computing in Python. *Nature Methods, 17*, 261–272. https://doi.org/10.1038/s41592-019-0686-2

Waskom, M. L. (2021). Seaborn: Statistical data visualization. *Journal of Open Source Software, 6*(60), 3021. https://doi.org/10.21105/joss.03021